# employee_silver_to_gold
Reads silver Delta table, runs all 4 analytics requirements, writes Gold Delta

In [0]:
%run /Workspace/Users/swati.k@diggibyte.com/databricks-assignment/src/Question_1/source_to_bronze/utils

In [0]:

silver_path = '/Volumes/workspace/default/upload/silver/Employee_info/dim_employee'

employee_df = spark.read.format('delta').load(silver_path)

print('Silver table loaded')
print('Row count:', employee_df.count())
print('Columns  :', employee_df.columns)
display(employee_df)

In [0]:

department_df = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load('/Volumes/workspace/default/upload/source_to_bronze/department')

country_df = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load('/Volumes/workspace/default/upload/source_to_bronze/country')

print('Department columns:', department_df.columns)  
print('Country columns   :', country_df.columns)     
display(department_df)
display(country_df)

In [0]:

emp_with_dept = employee_df.join(
    department_df,
    employee_df['department'] == department_df['DepartmentID'],
    'left'
)

print(' Joined with department')
print('Columns:', emp_with_dept.columns)
display(emp_with_dept)

In [0]:

emp_full_df = emp_with_dept.join(
    country_df,
    emp_with_dept['country'] == country_df['CountryCode'],
    'left'
)

print('Joined with country')
print('Columns:', emp_full_df.columns)
display(emp_full_df)

In [0]:

from pyspark.sql.functions import sum, count, avg, round, current_date

salary_df = emp_full_df.groupBy('DepartmentName') \
    .agg(sum('salary').alias('total_salary')) \
    .orderBy('total_salary', ascending=False) \
    .withColumn('at_load_date', current_date())

print('Requirement 1 — Total salary per department (descending):')
display(salary_df)

In [0]:

emp_count_df = emp_full_df.groupBy('DepartmentName', 'CountryName') \
    .agg(count('employee_id').alias('employee_count')) \
    .orderBy('DepartmentName', 'CountryName') \
    .withColumn('at_load_date', current_date())

print('Requirement 2 — Employee count per department per country:')
display(emp_count_df)

In [0]:

dept_country_df = emp_full_df.select('DepartmentName', 'CountryName') \
    .distinct() \
    .orderBy('DepartmentName') \
    .withColumn('at_load_date', current_date())

print('Requirement 3 — Department and Country names:')
display(dept_country_df)

In [0]:

avg_age_df = emp_full_df.groupBy('DepartmentName') \
    .agg(round(avg('age'), 2).alias('avg_age')) \
    .orderBy('DepartmentName') \
    .withColumn('at_load_date', current_date())

print(' Requirement 4 — Average age per department:')
display(avg_age_df)

In [0]:

fact_employee_df = emp_full_df \
    .join(salary_df.select('DepartmentName', 'total_salary'),
          'DepartmentName', 'left') \
    .join(emp_count_df.select('DepartmentName', 'CountryName', 'employee_count'),
          ['DepartmentName', 'CountryName'], 'left') \
    .join(avg_age_df.select('DepartmentName', 'avg_age'),
          'DepartmentName', 'left') \
    .select(
        'employee_id',
        'employee_name',
        'DepartmentName',
        'CountryName',
        'salary',
        'total_salary',
        'age',
        'avg_age',
        'employee_count',
        'load_date'
    ) \
    .withColumn('at_load_date', current_date())

print('fact_employee_df built')
print('Row count:', fact_employee_df.count())
display(fact_employee_df)

In [0]:

gold_path = '/Volumes/workspace/default/upload/gold/employee/fact_employee'

print(' Gold path:', gold_path)

In [0]:

from datetime import date

today = str(date.today()) 

fact_employee_df.write \
    .format('delta') \
    .mode('overwrite') \
    .option('replaceWhere', f"at_load_date = '{today}'") \
    .save(gold_path)

print(f'Gold table written (replaceWhere at_load_date = {today})')

In [0]:

gold_df = spark.read.format('delta').load(gold_path)

print('Gold table verified')
print('Row count:', gold_df.count())
display(gold_df)

In [0]:

display(dbutils.fs.ls('/Volumes/workspace/default/upload/gold/employee'))